# Chapter 17 Companion Notebook: Reinforcement Learning in Finance

This notebook reproduces Chapter 17's worked example: reformulating Chapter 5's American put early-exercise decision (the two-step binomial tree, S0=K=$50, u=1.2, d=0.9, R=1.04) as a three-state Markov decision process, solving it first by backward induction (Section 1, matching Chapter 5's closed-form values exactly), and then recovering the same solution using model-free tabular Q-learning (Section 2), comparing the learned and closed-form Q-values directly (Section 3).

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. The MDP formalization and closed-form (backward-induction) solution

In [1]:
import numpy as np

# Chapter 5's exact binomial tree parameters (American put, S0=K=$50, u=1.2, d=0.9, R=1.04)
S0, K, u, d, R = 50.0, 50.0, 1.2, 0.9, 1.04
q = (R - d) / (u - d)
gamma = 1.0 / R
print(f"risk-neutral up-probability q = {q:.4f}")
print(f"per-step discount factor gamma = {gamma:.4f}")

# terminal payoffs (American put, max(K-S,0)) at the three t=2 nodes
Suu, Sud, Sdd = S0*u*u, S0*u*d, S0*d*d
pay_uu, pay_ud, pay_dd = max(K-Suu,0), max(K-Sud,0), max(K-Sdd,0)
print(f"\nterminal nodes: Suu={Suu}, Sud={Sud}, Sdd={Sdd}")
print(f"terminal payoffs: uu={pay_uu}, ud={pay_ud}, dd={pay_dd}")

# closed-form backward induction (this is a Bellman optimality equation solved by hand)
Q_S1u_hold = gamma * (q*pay_uu + (1-q)*pay_ud)
Q_S1u_ex = max(K - S0*u, 0)
Q_S1d_hold = gamma * (q*pay_ud + (1-q)*pay_dd)
Q_S1d_ex = max(K - S0*d, 0)
V_S1u = max(Q_S1u_hold, Q_S1u_ex)
V_S1d = max(Q_S1d_hold, Q_S1d_ex)
Q_S0_hold = gamma * (q*V_S1u + (1-q)*V_S1d)
Q_S0_ex = max(K - S0, 0)

print("\nClosed-form (backward induction) Q-values:")
print(f"  Q(S0, exercise)  = {Q_S0_ex:.4f}   Q(S0, hold)  = {Q_S0_hold:.4f}")
print(f"  Q(Su, exercise)  = {Q_S1u_ex:.4f}   Q(Su, hold)  = {Q_S1u_hold:.4f}")
print(f"  Q(Sd, exercise)  = {Q_S1d_ex:.4f}   Q(Sd, hold)  = {Q_S1d_hold:.4f}")
print(f"\nAmerican put value V*(S0) = max(Q_S0_ex, Q_S0_hold) = {max(Q_S0_ex, Q_S0_hold):.4f}")
print("(matches Chapter 5's own American put value of approximately $2.5641 exactly)")

risk-neutral up-probability q = 0.4667
per-step discount factor gamma = 0.9615

terminal nodes: Suu=72.0, Sud=54.0, Sdd=40.5
terminal payoffs: uu=0, ud=0, dd=9.5

Closed-form (backward induction) Q-values:
  Q(S0, exercise)  = 0.0000   Q(S0, hold)  = 2.5641
  Q(Su, exercise)  = 0.0000   Q(Su, hold)  = 0.0000
  Q(Sd, exercise)  = 5.0000   Q(Sd, hold)  = 4.8718

American put value V*(S0) = max(Q_S0_ex, Q_S0_hold) = 2.5641
(matches Chapter 5's own American put value of approximately $2.5641 exactly)


## 2. Model-free tabular Q-learning (no knowledge of q used in the update rule)

### A worked example: the first two learning steps by hand, before running the full simulation

In [2]:
alpha_demo = 0.1
Q_demo = {"S0": {"exercise": 0.0, "hold": 0.0}, "S1d": {"exercise": 0.0, "hold": 0.0}}

# Step 1: at S1d, action "hold", simulated down-move -> terminal payoff pay_dd, episode ends
target_1 = pay_dd + gamma * 0.0
Q_demo["S1d"]["hold"] += alpha_demo * (target_1 - Q_demo["S1d"]["hold"])
print(f"Step 1: Q(S1d, hold) <- 0 + {alpha_demo}*[{pay_dd} + gamma*0 - 0] = {Q_demo['S1d']['hold']:.4f}")

# Step 2: at S0, action "hold", simulated down-move -> transitions to S1d, reward 0 (not terminal)
best_next_S1d = max(Q_demo["S1d"].values())
target_2 = 0.0 + gamma * best_next_S1d
Q_demo["S0"]["hold"] += alpha_demo * (target_2 - Q_demo["S0"]["hold"])
print(f"Step 2: Q(S0, hold) <- 0 + {alpha_demo}*[0 + gamma*{best_next_S1d:.4f} - 0] = {Q_demo['S0']['hold']:.4f}")

print(f"\nFor comparison, the closed-form targets are Q(S1d,hold)={Q_S1d_hold:.4f} and Q(S0,hold)={Q_S0_hold:.4f}:")
print("two updates already show the right qualitative shape, though both are still far from converged.")

Step 1: Q(S1d, hold) <- 0 + 0.1*[9.5 + gamma*0 - 0] = 0.9500
Step 2: Q(S0, hold) <- 0 + 0.1*[0 + gamma*0.9500 - 0] = 0.0913

For comparison, the closed-form targets are Q(S1d,hold)=4.8718 and Q(S0,hold)=2.5641:
two updates already show the right qualitative shape, though both are still far from converged.


In [3]:
rng = np.random.default_rng(7)

states = ["S0", "S1u", "S1d"]
actions = ["exercise", "hold"]
Qtab = {s: {a: 0.0 for a in actions} for s in states}
visits = {s: {a: 0 for a in actions} for s in states}

epsilon0 = 0.30
n_episodes = 100_000

def act_eps_greedy(s, ep):
    eps = max(0.02, epsilon0 * (1 - ep / n_episodes))
    if rng.random() < eps:
        return rng.choice(actions)
    qa = Qtab[s]
    return "exercise" if qa["exercise"] >= qa["hold"] else "hold"

def update(s, a, target):
    visits[s][a] += 1
    alpha = 1.0 / (1 + visits[s][a]) ** 0.6
    Qtab[s][a] += alpha * (target - Qtab[s][a])

for ep in range(n_episodes):
    s = "S0"
    a = act_eps_greedy(s, ep)
    if a == "exercise":
        update(s, a, Q_S0_ex)
        continue
    else:
        s_next = "S1u" if rng.random() < q else "S1d"
        best_next = max(Qtab[s_next].values())
        update(s, a, 0.0 + gamma * best_next)
        s = s_next

    a = act_eps_greedy(s, ep)
    if a == "exercise":
        r = Q_S1u_ex if s == "S1u" else Q_S1d_ex
        update(s, a, r)
    else:
        if s == "S1u":
            payoff = pay_uu if rng.random() < q else pay_ud
        else:
            payoff = pay_ud if rng.random() < q else pay_dd
        update(s, a, 0.0 + gamma * payoff)

print(f"Learned Q-table after {n_episodes:,} simulated episodes (seed=7):\n")
for s in states:
    print(f"  {s}: exercise={Qtab[s]['exercise']:.4f}  hold={Qtab[s]['hold']:.4f}   (visits: {visits[s]})")

print("\nLearned greedy policy:")
for s in states:
    best_a = max(Qtab[s], key=Qtab[s].get)
    print(f"  {s} -> {best_a}")

Learned Q-table after 100,000 simulated episodes (seed=7):

  S0: exercise=0.0000  hold=2.5328   (visits: {'exercise': 7572, 'hold': 92428})
  S1u: exercise=0.0000  hold=0.0000   (visits: {'exercise': 39970, 'hold': 3127})
  S1d: exercise=5.0000  hold=4.9234   (visits: {'exercise': 44604, 'hold': 4727})

Learned greedy policy:
  S0 -> hold
  S1u -> exercise
  S1d -> exercise


## 3. Comparing the learned and closed-form solutions

In [4]:
closed_form = {
    ("S0", "exercise"): Q_S0_ex, ("S0", "hold"): Q_S0_hold,
    ("S1u", "exercise"): Q_S1u_ex, ("S1u", "hold"): Q_S1u_hold,
    ("S1d", "exercise"): Q_S1d_ex, ("S1d", "hold"): Q_S1d_hold,
}

print(f"{'state-action':<18}{'closed-form':>14}{'learned':>14}{'rel. gap':>12}")
for s in states:
    for a in actions:
        cf = closed_form[(s, a)]
        learned = Qtab[s][a]
        gap = abs(cf - learned) / cf if cf > 0 else 0.0
        gap_str = f"{gap*100:.2f}%" if cf > 0 else "---"
        print(f"({s}, {a:<9}){cf:>13.4f} {learned:>13.4f} {gap_str:>12}")

print("\nThe learned policy matches the closed-form optimal policy: hold at S0 and (tie) at S1u,")
print("exercise at S1d. Every 'exercise' value converges exactly (deterministic, one-step reward);")
print("every 'hold' value converges to within roughly 1% of the true value, the expected residual")
print("noise from estimating a continuation value via sampled transitions rather than the exact")
print("risk-neutral probability q.")

state-action         closed-form       learned    rel. gap
(S0, exercise )       0.0000        0.0000          ---
(S0, hold     )       2.5641        2.5328        1.22%
(S1u, exercise )       0.0000        0.0000          ---
(S1u, hold     )       0.0000        0.0000          ---
(S1d, exercise )       5.0000        5.0000        0.00%
(S1d, hold     )       4.8718        4.9234        1.06%

The learned policy matches the closed-form optimal policy: hold at S0 and (tie) at S1u,
exercise at S1d. Every 'exercise' value converges exactly (deterministic, one-step reward);
every 'hold' value converges to within roughly 1% of the true value, the expected residual
noise from estimating a continuation value via sampled transitions rather than the exact
risk-neutral probability q.


## 4. A larger worked example: Q-learning recovers the Almgren-Chriss optimal execution trajectory (Chapter 11)

In [5]:
# Chapter 11's exact Almgren-Chriss worked-example parameters
X_TOTAL, N, ETA, SIGMA, LAMBDA, TAU, GRID = 250.0, 5, 0.01, 0.02, 0.5, 1.0, 5.0
NMAX = int(X_TOTAL / GRID)  # 50 grid units of 5 shares each

# sanity check: reproduce Chapter 11's continuous closed-form cost of 139.4
x_cont = np.array([250, 194.2, 142.4, 93.4, 46.2, 0])
a_cont = -np.diff(x_cont)
impact_cost = ETA * np.sum(a_cont**2) / TAU
timing_cost = LAMBDA * SIGMA**2 * TAU * np.sum(x_cont[1:]**2)
print(f"Continuous closed-form check: total cost = {impact_cost + timing_cost:.4f} (Chapter 11 reports 139.4)")

def cost_impact(m):
    return ETA * (GRID * m) ** 2 / TAU

def cost_timing(n_after):
    return LAMBDA * SIGMA ** 2 * TAU * (GRID * n_after) ** 2

def cost_forced_liquidation(n_after):
    return ETA * (GRID * n_after) ** 2

# exact backward-induction DP over the discretized grid
def backward_stage(Vnext, forced_next=False):
    V = np.full(NMAX + 1, np.inf)
    A = np.zeros(NMAX + 1, dtype=int)
    for n in range(NMAX + 1):
        best, best_m = np.inf, 0
        for m in range(n + 1):
            n_after = n - m
            c = cost_impact(m) + cost_timing(n_after)
            total = c + cost_forced_liquidation(n_after) if forced_next else c + Vnext[n_after]
            if total < best:
                best, best_m = total, m
        V[n], A[n] = best, best_m
    return V, A

V4, A4 = backward_stage(None, forced_next=True)
V3, A3 = backward_stage(V4)
V2, A2 = backward_stage(V3)
V1, A1 = backward_stage(V2)
DP_TABLES = {1: A1, 2: A2, 3: A3, 4: A4}

n = NMAX
dp_traj = [n]
for j in [1, 2, 3, 4]:
    m = DP_TABLES[j][n]
    n -= m
    dp_traj.append(n)
dp_traj.append(0)

print("\nExact DP-optimal discretized trajectory (shares):", [GRID * v for v in dp_traj])
print("Exact DP-optimal total cost:", round(V1[NMAX], 2))

Continuous closed-form check: total cost = 139.3715 (Chapter 11 reports 139.4)

Exact DP-optimal discretized trajectory (shares): [250.0, np.float64(195.0), np.float64(145.0), np.float64(95.0), np.float64(45.0), 0.0]
Exact DP-optimal total cost: 139.52


In [6]:
# tabular Q-learning over the same deterministic MDP, never given the DP tables above
rng2 = np.random.default_rng(11)
NEG = -1e18
Q = {j: np.full((NMAX + 1, NMAX + 1), NEG) for j in [1, 2, 3, 4]}
visits = {j: np.zeros((NMAX + 1, NMAX + 1), dtype=np.int64) for j in [1, 2, 3, 4]}
for j in [1, 2, 3, 4]:
    for n_ in range(NMAX + 1):
        Q[j][n_, :n_ + 1] = 0.0

N_EPISODES = 600_000
EPS0 = 0.5

def choose_action(j, n_, ep):
    eps = max(0.05, EPS0 * (1 - ep / N_EPISODES))
    if rng2.random() < eps:
        return rng2.integers(0, n_ + 1)
    return int(np.argmax(Q[j][n_, :n_ + 1]))

for ep in range(N_EPISODES):
    n_ = NMAX
    for j in [1, 2, 3, 4]:
        m = choose_action(j, n_, ep)
        n_after = n_ - m
        if j < 4:
            reward = -(cost_impact(m) + cost_timing(n_after))
        else:
            reward = -(cost_impact(m) + cost_timing(n_after) + cost_forced_liquidation(n_after))
        visits[j][n_, m] += 1
        alpha = 1.0 / (1 + visits[j][n_, m]) ** 0.55
        target = reward + (np.max(Q[j + 1][n_after, :n_after + 1]) if j < 4 else 0.0)
        Q[j][n_, m] += alpha * (target - Q[j][n_, m])
        n_ = n_after

n_ = NMAX
learned_traj = [n_]
for j in [1, 2, 3, 4]:
    m = int(np.argmax(Q[j][n_, :n_ + 1]))
    n_ -= m
    learned_traj.append(n_)
learned_traj.append(0)

print(f"Trained {N_EPISODES:,} episodes.")
print("Learned greedy trajectory (shares):", [GRID * v for v in learned_traj])

Trained 600,000 episodes.
Learned greedy trajectory (shares): [250.0, 195.0, 145.0, 95.0, 45.0, 0.0]


In [7]:
def trajectory_cost(traj_n):
    total = 0.0
    for j in range(1, 4):
        n_before, n_after = traj_n[j - 1], traj_n[j]
        m = n_before - n_after
        total += cost_impact(m) + cost_timing(n_after)
    n_before, n_after = traj_n[3], traj_n[4]
    m = n_before - n_after
    total += cost_impact(m) + cost_timing(n_after) + cost_forced_liquidation(traj_n[4])
    return total

learned_cost = trajectory_cost(learned_traj[:5])
dp_cost = trajectory_cost(dp_traj[:5])
print(f"Learned trajectory's true cost:  {learned_cost:.2f}")
print(f"DP-optimal trajectory's cost:    {dp_cost:.2f}")
print(f"Match: {learned_traj == dp_traj}")

twap = [50, 40, 30, 20, 10, 0]  # in grid units (250,200,150,100,50,0 shares)
aggressive = [50, 0, 0, 0, 0, 0]
print(f"\nUniform TWAP cost:        {trajectory_cost(twap):.2f}")
print(f"Fully aggressive cost:    {trajectory_cost(aggressive):.2f}")
print(f"Q-learning (learned) cost: {learned_cost:.2f}")

Learned trajectory's true cost:  139.52
DP-optimal trajectory's cost:    139.52
Match: True

Uniform TWAP cost:        140.00
Fully aggressive cost:    625.00
Q-learning (learned) cost: 139.52


## 5. Portfolio management: Q-learning versus direct estimation for the Kelly fraction (Chapter 1)

In [8]:
# Chapter 1's exact Kelly criterion parameters: even-money bet, 60% win probability
P_WIN, B = 0.6, 1.0
Q_LOSE = 1 - P_WIN

def growth_rate(f):
    return P_WIN * np.log(1 + B * f) + Q_LOSE * np.log(1 - f)

f_star = P_WIN - Q_LOSE / B
grid = np.round(np.arange(0.0, 1.0, 0.05), 2)
g_grid = np.array([growth_rate(f) for f in grid])
n_actions = len(grid)

print(f"Closed-form f* = {f_star:.2f}, g(f*) = {growth_rate(f_star):.4f}")
print(f"Nearest grid neighbors: g(0.15)={growth_rate(0.15):.4f}, g(0.25)={growth_rate(0.25):.4f}")
print(f"Gap to nearest neighbor: {growth_rate(f_star) - growth_rate(0.15):.4f}")
print(f"Single-bet outcomes: win={np.log(1.2):.4f}, lose={np.log(0.8):.4f} (huge relative to that gap)")

# stateless tabular Q-learning (bandit-style: no bootstrapping, one-shot reward each period)
rng3 = np.random.default_rng(3)
Q = np.zeros(n_actions)
visits = np.zeros(n_actions, dtype=np.int64)
total_wins, total_bets = 0, 0

N_EPISODES = 200_000
EPS0 = 0.4

for ep in range(N_EPISODES):
    eps = max(0.02, EPS0 * (1 - ep / N_EPISODES))
    if rng3.random() < eps:
        a = rng3.integers(0, n_actions)
    else:
        a = np.argmax(Q)
    f = grid[a]
    win = rng3.random() < P_WIN
    total_wins += int(win)
    total_bets += 1
    r = np.log(1 + B * f) if win else np.log(1 - f)
    visits[a] += 1
    alpha = 1.0 / (1 + visits[a]) ** 0.6
    Q[a] += alpha * (r - Q[a])

learned_best = grid[np.argmax(Q)]
shortfall = growth_rate(f_star) - growth_rate(learned_best)
print(f"\nAfter {N_EPISODES:,} simulated bets:")
print(f"  Q-learning's learned-best fraction: {learned_best:.2f}")
print(f"  Shortfall vs true optimum: {shortfall:.4f} ({shortfall/growth_rate(f_star)*100:.1f}% of optimal growth rate)")

# model-based: estimate p directly from the SAME data and plug into the closed form
p_hat = total_wins / total_bets
f_hat_model_based = p_hat - (1 - p_hat) / B
print(f"\nModel-based estimate from the same {total_bets:,} observations:")
print(f"  observed win frequency p-hat = {p_hat:.4f} (true p = {P_WIN})")
print(f"  f-hat* = p-hat - (1-p-hat)/b = {f_hat_model_based:.4f} (true f* = {f_star:.2f})")
print(f"  model-based error: {abs(f_hat_model_based - f_star):.4f}  (vs. Q-learning's error of {abs(learned_best - f_star):.2f})")

Closed-form f* = 0.20, g(f*) = 0.0201
Nearest grid neighbors: g(0.15)=0.0188, g(0.25)=0.0188
Gap to nearest neighbor: 0.0013
Single-bet outcomes: win=0.1823, lose=-0.2231 (huge relative to that gap)



After 200,000 simulated bets:
  Q-learning's learned-best fraction: 0.15
  Shortfall vs true optimum: 0.0013 (6.4% of optimal growth rate)

Model-based estimate from the same 200,000 observations:
  observed win frequency p-hat = 0.6003 (true p = 0.6)
  f-hat* = p-hat - (1-p-hat)/b = 0.2006 (true f* = 0.20)
  model-based error: 0.0006  (vs. Q-learning's error of 0.05)
